# 12 - Qwen3 Reranker 8B Evaluation

Heavy reranker ablation with `Qwen/Qwen3-Reranker-8B`. It reranks top-30 dense candidates and reports the same retrieval metrics. Run notebook 11 first if you want to rerank Qwen3-Embedding-8B candidates; otherwise this falls back to the existing BGE-M3 dense index.

In [ ]:
!pip install -q -U "sentence-transformers>=5.1.0" "transformers>=4.51.0" accelerate faiss-cpu rank-bm25

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
qwen_index_root = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b'
bge_index_root = DRIVE_ROOT / config['official_index_root']

if (qwen_index_root / 'index_manifest.json').exists():
    index_root = qwen_index_root
    candidate_label = 'qwen3_embedding_8b_dense_top30'
else:
    index_root = bge_index_root
    candidate_label = 'bge_m3_dense_top30'

index_root, candidate_label

In [ ]:
from src.evaluation_reranker import evaluate_reranker

reranker_model = 'Qwen/Qwen3-Reranker-8B'
benchmark_csv = DRIVE_ROOT / config['benchmark_csv']

summary = evaluate_reranker(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_predictions_csv=DRIVE_ROOT / f'outputs/retrieval_eval/{candidate_label}_qwen3_reranker_8b_predictions_v1.csv',
    output_summary_json=DRIVE_ROOT / f'outputs/retrieval_eval/{candidate_label}_qwen3_reranker_8b_summary_v1.json',
    candidate_mode='dense',
    candidate_k=30,
    top_k=10,
    reranker_model=reranker_model,
    batch_size=1,
    device=device,
)
summary

In [ ]:
import pandas as pd

rows = []
baseline_summary_path = DRIVE_ROOT / 'outputs/retrieval_eval/dense_retrieval_summary_v1.json'
if baseline_summary_path.exists():
    baseline = json.loads(baseline_summary_path.read_text(encoding='utf-8'))
    rows.append({'experiment': 'dense candidates before rerank', **baseline['metrics']})
rows.append({'experiment': f'{candidate_label}+Qwen3-Reranker-8B', **summary['metrics']})
pd.DataFrame(rows)[[
    'experiment', 'doc_hit@5', 'doc_hit@10', 'article_hit@5', 'article_hit@10',
    'doc_mrr', 'article_mrr', 'article_ndcg@5', 'article_ndcg@10'
]]